# Code for Task 2 : Interactive Debugging Tool

Ressources used:

https://ipywidgets.readthedocs.io/en/latest/user_install.html

https://www.geeksforgeeks.org/python/read-json-file-using-python/ 

https://www.geeksforgeeks.org/python/plot-a-point-or-a-line-on-an-image-with-matplotlib/ 

https://pynative.com/python-check-if-key-exists-in-json-and-iterate-the-json-array/ 


In [ ]:
# To change depending on how the raw data will be saved as
# here im supposing a separate file for each play
# plays saved in a folder by year/season

#lets say i want the game 2022030411
#it would be in the folder 2022

# implementation done using file ../data/2022030411_play-by-play.json as reference

import json

#Function to get json file and convert it to python object
def get_info(game: str):
    year = game[0:4]

    #define the path to the stored data
    filepath = "../data"

    #to change depending how how the files will be saved as
    filepath += "/" + year + "/" + game + "_play-by-play.json"

    #Read JSON file
    with open(filepath,"r") as file:
        data = json.load(file)
    

    return data


# Function to get the game data
def game_data(year,type,id):
    gameType = ""
    if type == 'Regular season':
        gameType = "02"
    elif type == 'Playoffs':
        gameType = "03"
    else:
        print("Not a game from regular season or playoffs.")

    strGameId = str(id)
    if len(strGameId) != 4:
        strGameId = "0" + str(id)

    game = str(year) + gameType + strGameId

    return get_info(game)


# Function to interact with widgets
# Print information about the game
# Draw the coordinates on the skating rink image
def skating_rink(y,t,g,e):
    year = y
    gameId = g
    matchType = t
    eventId  = e

    data = game_data(year,matchType,gameId)
    #Time, game ID, home vs away, stats (teams,goals,SoG, SO goals, SO attempts)
    home = data["homeTeam"]["abbrev"]
    away = data["awayTeam"]["abbrev"]

    print(data["startTimeUTC"])
    print("Game ID: " + str(gameId) + "; " + home + " (home) vs " + away + " (away)")

    events = data["plays"]


    #Search information about eventId
    infoEvent = {}
    for event in events:
        if(event["eventId"]==eventId):
            infoEvent = event

    img = image.imread('../../figures/nhl_rink.png')

    description = "event" #Changes for each event

    #coordinates change fo each event
    if "details" in infoEvent:
        x = infoEvent["details"]["xCoord"]
        y = infoEvent["details"]["yCoord"]

        plt.plot(x,y,marker='o',color='blue') 
        plt.title(description) 
        plt.imshow(img,extent=[-100,100,-42.5,42.5])
        plt.show()

    #print info about the event
    print(infoEvent)
    return

Game IDs (https://gitlab.com/dword4/nhlapi/-/blob/master/stats-api.md#game-ids)

will look like this: 2023020001
The first 4 digits identify the season of the game (ie. 2017 for the 2017-2018 season). Always refer to a season with the starting year. A game played in March 2018 would still have a game ID that starts with 2017

The next 2 digits give the type of game, where 01 = preseason, 02 = regular season, 03 = playoffs, 04 = all-star

The final 4 digits identify the specific game number. 

For regular season and preseason games, this ranges from 0001 to the number of games played. (1353 for seasons with 32 teams (2022 - Present), 1271 for seasons with 31 teams (2017 - 2020) and 1230 for seasons with 30 teams). 

For playoff games, the 2nd digit of the specific number gives the round of the playoffs, the 3rd digit specifies the matchup, and the 4th digit specifies the game (out of 7).

In [3]:
# ipywidgets
import ipywidgets as widgets
from ipywidgets import interact

#Use matplotlib to draw on the skating rink image
from matplotlib import image
from matplotlib import pyplot as plt

In [ ]:
# Adding a widget to choose the year/season
wYear = widgets.Dropdown(
    options=[2016,2017,2018,2019,2020,2021,2022,2023],
    value=2022,
    description='Year/Season',
    disabled=False
)

# widget to choose the match type
wMatchType = widgets.Dropdown(
    options=['Regular season','Playoffs'],
    value='Playoffs',
    description='Match type',
    disabled=False
)

matchType = wMatchType.value

gameType = ""
if matchType == 'Regular season':
    gameType = "02"
elif matchType == 'Playoffs':
    gameType = "03"
else:
    print("Not a game from regular season or playoffs.")

#widget to choose the game ID
if gameType == "02" :
    wGameId = widgets.IntSlider(
        value = 411,
        min = 0,
        max = 1353, #max possible games played for regular season
        step = 1,
        description = 'game_Id',
        disabled = False,
        continuous_update=False,
        orientation='horizontal',
        readout=True,
        readout_format='d'
    )
elif gameType == "03" :
    wGameId = widgets.IntSlider(
            value = 411,
            min = 0,
            max = 1000, #to change for max possible games played for playoffs
            step = 1,
            description = 'game_Id',
            disabled = False,
            continuous_update=False,
            orientation='horizontal',
            readout=True,
            readout_format='d'
        )

wGameId = widgets.IntSlider(
            value = 411,
            min = 0,
            max = 1000, #to change for max possible games played for playoffs
            step = 1,
            description = 'game_Id',
            disabled = False,
            continuous_update=False,
            orientation='horizontal',
            readout=True,
            readout_format='d'
        )

#print info about the game
# interact(game_info,y = wYear,t = wMatchType,g = wGameId) #INTERACT


#get events for the game
data = game_data(wYear.value,wMatchType.value,wGameId.value)
events = data["plays"]

maxEventId = 0
for event in events:
    if event["eventId"] > maxEventId:
        maxEventId = event["eventId"]

# widget to choose event ID
wEventId = widgets.IntSlider(
    value = 54,
    min = 0,
    max = maxEventId,
    step = 1,
    description = 'event_Id',
    disabled = False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

interact(skating_rink,y = wYear,t = wMatchType,g = wGameId,e=wEventId) #INTERACT


interactive(children=(Dropdown(description='Year/Season', index=6, options=(2016, 2017, 2018, 2019, 2020, 2021…

<function __main__.skating_rink(y, t, g, e)>

In [5]:
wYear.close()
wMatchType.close()
wGameId.close()
wEventId.close()